# EDA

In [1]:
import pandas as pd
import os
import gzip
from pathlib import Path
import hashlib

## Obtener direcciones de archivos

In [2]:
mainPath ='../data/raw/'
dirs = []
for item in os.listdir(mainPath):
    lclPath = Path(mainPath).joinpath(item)
    for f in lclPath.glob('*.csv'):
        dirs.append(str(f))

dirs

['..\\data\\raw\\2025\\datos_abiertos_2025_01.csv',
 '..\\data\\raw\\2025\\datos_abiertos_2025_02.csv',
 '..\\data\\raw\\2025\\datos_abiertos_2025_03.csv',
 '..\\data\\raw\\2025\\datos_abiertos_2025_04.csv',
 '..\\data\\raw\\2025\\datos_abiertos_2025_05.csv',
 '..\\data\\raw\\2025\\datos_abiertos_2025_06.csv',
 '..\\data\\raw\\2025\\datos_abiertos_2025_07.csv',
 '..\\data\\raw\\2025\\datos_abiertos_2025_08.csv',
 '..\\data\\raw\\2025\\datos_abiertos_2025_09.csv',
 '..\\data\\raw\\2025\\datos_abiertos_2025_10.csv',
 '..\\data\\raw\\2025\\datos_abiertos_2025_11.csv',
 '..\\data\\raw\\2025\\datos_abiertos_2025_12.csv',
 '..\\data\\raw\\2026\\datos_abiertos_2026_01.csv',
 '..\\data\\raw\\2026\\datos_abiertos_2026_02.csv',
 '..\\data\\raw\\2026\\datos_abiertos_2026_03.csv',
 '..\\data\\raw\\2026\\datos_abiertos_2026_04.csv',
 '..\\data\\raw\\2026\\datos_abiertos_2026_05.csv',
 '..\\data\\raw\\2026\\datos_abiertos_2026_06.csv',
 '..\\data\\raw\\2026\\datos_abiertos_2026_07.csv',
 '..\\data\\

## Unificación de CSV

In [3]:
dfs = [pd.read_csv(path, encoding='latin1') for path in dirs if str(path).__contains__('datos_abiertos')]

df = pd.concat(dfs, ignore_index=True)
df

,Viaje_Id,Usuario_Id,Genero,Año_de_nacimiento,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id,AÃ±o_de_nacimiento
0,37162342,601273,M,1982.0,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395,NaN
1,37162343,1571524,M,2003.0,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260,NaN
2,37162344,1435200,F,1984.0,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260,NaN
3,37162345,126327,M,1988.0,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11,NaN
4,37162346,416121,M,1994.0,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8,NaN
...,...,...,...,...,...,...,...,...,...
7894092,46255554,116275,F,NaN,2026-09-30 23:57:23,2026-09-30 23:59:46,43,44,1990.0
7894093,46255556,4621160,M,NaN,2026-09-30 23:59:13,2026-10-01 00:04:17,210,413,1970.0
7894094,46255556,4621160,M,NaN,2026-09-30 23:59:13,2026-10-01 00:04:17,210,413,1970.0
7894095,46255557,4560372,M,NaN,2026-09-30 23:59:32,2026-10-01 00:06:16,157,271,1987.0


## Eliminacion de columnas

Dado el objetivo de identificar la cantidad requerida por hora en los puntos de orige, hay campos que ofrecen nulo valor a la predicción. Los campos a mantener serian los tiempos de viaje (inicio y fin)  asi como el origen y destino.

In [4]:
# df['Año_de_nacimiento'] = (
#     df['Año_de_nacimiento']
#     .fillna(df['AÃ±o_de_nacimiento'])
# )

df.drop(columns=['AÃ±o_de_nacimiento','Año_de_nacimiento','Viaje_Id','Genero','Usuario_Id'], inplace=True)

df

,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395
1,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260
2,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260
3,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11
4,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8
...,...,...,...,...
7894092,2026-09-30 23:57:23,2026-09-30 23:59:46,43,44
7894093,2026-09-30 23:59:13,2026-10-01 00:04:17,210,413
7894094,2026-09-30 23:59:13,2026-10-01 00:04:17,210,413
7894095,2026-09-30 23:59:32,2026-10-01 00:06:16,157,271


## Eliminacion de duplicados

In [5]:
df.drop_duplicates(inplace=True)
df.reset_index(drop=True, inplace=True)
df

,Inicio_del_viaje,Fin_del_viaje,Origen_Id,Destino_Id
0,2025-01-01 00:00:44,2025-01-01 00:11:29,211,395
1,2025-01-01 00:04:11,2025-01-01 00:15:07,35,260
2,2025-01-01 00:05:27,2025-01-01 00:15:23,35,260
3,2025-01-01 00:07:47,2025-01-01 00:17:42,190,11
4,2025-01-01 00:08:52,2025-01-01 00:18:10,273,8
...,...,...,...,...
7891822,2026-09-30 23:56:38,2026-10-01 00:14:52,73,73
7891823,2026-09-30 23:57:23,2026-09-30 23:59:46,43,44
7891824,2026-09-30 23:59:13,2026-10-01 00:04:17,210,413
7891825,2026-09-30 23:59:32,2026-10-01 00:06:16,157,271


## Esquema inicial

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7891827 entries, 0 to 7891826
Data columns (total 4 columns):
 #   Column            Dtype 
---  ------            ----- 
 0   Inicio_del_viaje  object
 1   Fin_del_viaje     object
 2   Origen_Id         int64 
 3   Destino_Id        int64 
dtypes: int64(2), object(2)
memory usage: 240.8+ MB


## Conversion de tipos de dato

In [7]:
df['Destino_Id'] = df['Destino_Id'].astype('str')
df['Origen_Id'] = df['Origen_Id'].astype('str')
df['Inicio_del_viaje'] = pd.to_datetime(df['Inicio_del_viaje'],format="%Y-%m-%d %H:%M:%S")
df['Fin_del_viaje'] = pd.to_datetime(df['Fin_del_viaje'],format="%Y-%m-%d %H:%M:%S")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7891827 entries, 0 to 7891826
Data columns (total 4 columns):
 #   Column            Dtype         
---  ------            -----         
 0   Inicio_del_viaje  datetime64[ns]
 1   Fin_del_viaje     datetime64[ns]
 2   Origen_Id         object        
 3   Destino_Id        object        
dtypes: datetime64[ns](2), object(2)
memory usage: 240.8+ MB


## Verificacion de nulos

In [15]:
display(df.isna().sum())
df.isnull().sum()

Inicio_del_viaje    0
Fin_del_viaje       0
Origen_Id           0
Destino_Id          0
dtype: int64

Inicio_del_viaje    0
Fin_del_viaje       0
Origen_Id           0
Destino_Id          0
dtype: int64

## Exploración de datos

### Frecuencia de ubicacion de origen por dia

In [8]:
origenes_pivot = (
    df.groupby([
        df['Inicio_del_viaje'].dt.date,
        'Origen_Id'
    ])
    .size()
    .unstack(fill_value=0)
)

origenes_pivot.head()

Origen_Id,10,100,101,102,103,104,105,106,107,108,...,87,88,9,91,92,93,94,95,96,98
Inicio_del_viaje,,,,,,,,,,,,,,,,,,,,,
2025-01-01,12,5,3,7,3,0,4,6,1,7,...,8,6,5,1,4,4,2,0,7,8
2025-01-02,33,10,3,24,11,0,12,18,7,31,...,21,24,35,1,12,7,11,9,20,8
2025-01-03,34,16,2,31,20,0,12,12,3,28,...,16,30,34,4,16,11,13,6,21,13
2025-01-04,32,3,2,20,6,0,13,11,3,8,...,20,16,25,5,10,12,7,5,11,8
2025-01-05,26,4,0,23,4,0,8,14,8,14,...,18,16,15,9,8,10,12,3,12,8


### Frecuencia de ubicacion de destino por dia

In [9]:
destinos_pivot = (
    df.groupby([
        df['Inicio_del_viaje'].dt.date,
        'Destino_Id'
    ])
    .size()
    .unstack(fill_value=0)
)

destinos_pivot.head()

Destino_Id,10,100,101,102,103,104,105,106,107,108,...,87,88,9,91,92,93,94,95,96,98
Inicio_del_viaje,,,,,,,,,,,,,,,,,,,,,
2025-01-01,7,4,2,6,3,0,4,4,1,11,...,1,4,9,3,2,4,8,1,8,5
2025-01-02,39,8,1,27,17,0,14,14,5,31,...,17,15,36,2,10,6,16,6,12,9
2025-01-03,36,13,3,23,12,0,12,19,3,20,...,15,14,31,7,14,9,15,6,11,8
2025-01-04,38,5,3,17,8,0,8,6,2,3,...,12,12,35,4,8,9,10,3,12,7
2025-01-05,26,2,1,22,7,0,9,9,6,14,...,10,14,13,5,4,9,17,4,8,7


### Frecuencia origen-destino

In [12]:
od = (
    df.groupby([
        df['Inicio_del_viaje'].dt.hour,
        'Origen_Id',
        'Destino_Id'
    ])
    .size()
    .reset_index(name='viajes')
)

od.head()

,Inicio_del_viaje,Origen_Id,Destino_Id,viajes
0,0,10,10,15
1,0,10,12,2
2,0,10,151,1
3,0,10,156,3
4,0,10,157,2
